# Mamba-SSM Oracle — single diagnostic comparison

**Project: `transformer-vs-ssm` (sequence-modeling blog series). Unrelated to any competition.**

## Why this exists

After fixing three real bugs in the from-scratch Mamba (`dt_proj` init, a blanket
reinit that erased it, and the missing `dt_rank` bottleneck), plus proving scan
parity against the official reference (`tests/test_selective_scan_parity.py`,
passes), the corrected model **still fails Selective Copying** at `ssm_lr_scale=1`
(commit `58ebc92`) -- flat loss at ln(64) through 800 steps, val_acc 0.008-0.012
vs. chance 0.032.

Per external review: when scan parity passes and the model still fails, the next
discriminating experiment is the **official `mamba-ssm` package as an oracle** --
same task, same dimensions, same optimizer/schedule -- to find out whether the
remaining gap is (a) still something specific to our implementation, or (b) the
task/training recipe itself at this scale/step budget.

## Decision table

| Custom Mamba | Official `mamba-ssm` | Interpretation |
|---|---|---|
| fails | **learns** | model-level discrepancy remains (wrapper/precision/training path) |
| fails | **fails** | stop debugging the architecture -- investigate task/recipe first |
| learns | learns | custom implementation validated |
| learns | fails | harness/config mismatch -- don't interpret either yet |

This notebook trains **three models, one seed each, same task/steps**, for a clean
comparison in one environment: Transformer (known-good control), our custom Mamba
(post-fix), official `mamba_ssm.modules.mamba_simple.Mamba` (Mamba-1, matching
what we've been debugging -- not Mamba-2).

## Kaggle settings
- Accelerator: **GPU T4 x1**, Internet **ON**
- Save Version -> Run All
- If `mamba-ssm` fails to install/import, the notebook stops with a clear
  diagnostic rather than failing opaquely later.

In [ ]:
# Cell 1: Clone (or refresh) the repo -- picks up the ablation fix (7ec4346)
import os, subprocess, sys, platform

REPO = "https://github.com/nvaidyan1/transformer-vs-ssm.git"
if not os.path.exists("transformer-vs-ssm"):
    subprocess.run(["git", "clone", REPO], check=True)

os.chdir("/kaggle/working/transformer-vs-ssm")
subprocess.run(["git", "pull", "--ff-only"], check=False)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

import torch
print(f"\npython {platform.python_version()}  torch {torch.__version__}  "
      f"cuda {torch.version.cuda}  "
      f"gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

In [ ]:
# Cell 2: Repo dependencies + full test suite (must pass before spending GPU time)
!pip install -q pyyaml pytest einops 2>&1 | tail -1
!python -m pytest tests/ -q 2>&1 | tail -6

In [ ]:
# Cell 3: Install mamba-ssm. causal-conv1d is optional (best-effort acceleration
# for the depthwise conv; the module falls back to plain PyTorch conv without it).
# mamba-ssm itself is NOT optional -- its compiled selective_scan extension is
# imported unconditionally by mamba_simple.py, so a failed build here blocks the
# whole notebook. pip tries a matching prebuilt wheel first and only falls back
# to a source build if none matches this exact torch/cuda/python combination.
import subprocess

print("Attempting causal-conv1d (optional)...")
r = subprocess.run(["pip", "install", "-q", "causal-conv1d>=1.4.0"],
                   capture_output=True, text=True)
print("  OK" if r.returncode == 0 else f"  not available (non-fatal): {r.stderr[-500:]}")

print("\nAttempting mamba-ssm (required)...")
r = subprocess.run(["pip", "install", "-q", "mamba-ssm"], capture_output=True, text=True)
if r.returncode != 0:
    print("=" * 70)
    print("mamba-ssm FAILED TO INSTALL. Full output below. The oracle experiment")
    print("cannot proceed on this environment -- stop here rather than fail later.")
    print("=" * 70)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    raise SystemExit("mamba-ssm install failed -- see diagnostic above")
print("  OK")

try:
    from mamba_ssm.modules.mamba_simple import Mamba as OfficialMamba
    import mamba_ssm
    print(f"\nmamba_ssm {mamba_ssm.__version__} imported OK")
except Exception as e:
    print("=" * 70)
    print(f"mamba-ssm installed but failed to IMPORT: {type(e).__name__}: {e}")
    print("Likely a torch/cuda ABI mismatch between the installed wheel and this")
    print("environment. Stop here -- see the version print in Cell 1.")
    print("=" * 70)
    raise

In [ ]:
# Cell 4: Oracle wrapper -- residual pre-norm block around the official Mamba-1
# mixer, matching our own MambaBlock structure (x + mixer(norm(x))) and our
# model's forward(x) -> logits interface, so train/eval code can be shared.
import torch
import torch.nn as nn

class OracleBlock(nn.Module):
    def __init__(self, d_model, d_state, d_conv, expand, layer_idx):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mixer = OfficialMamba(d_model=d_model, d_state=d_state, d_conv=d_conv,
                                   expand=expand, layer_idx=layer_idx)

    def forward(self, x):
        return x + self.mixer(self.norm(x))

class OracleMamba(nn.Module):
    def __init__(self, vocab_size, n_layers, d_model, d_state, d_conv, expand):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            OracleBlock(d_model, d_state, d_conv, expand, layer_idx=i)
            for i in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying, matching our model

    def forward(self, x):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln_f(h))

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
# Cell 5: Shared settings -- identical across all three models
TASK_ARGS = dict(n_data=4, seq_len=64, vocab_size=64)
D_MODEL, D_STATE, D_CONV, EXPAND, N_LAYERS = 124, 16, 4, 2, 4  # matches mamba_small.yaml
BATCH, LR, WARMUP, STEPS, SEED = 32, 3e-4, 200, 2000, 0
OUT = "/kaggle/working/oracle"

import os
os.makedirs(OUT, exist_ok=True)
print(f"task={TASK_ARGS}  d_model={D_MODEL} d_state={D_STATE} d_conv={D_CONV} "
      f"expand={EXPAND} n_layers={N_LAYERS}\n"
      f"batch={BATCH} lr={LR} warmup={WARMUP} steps={STEPS} seed={SEED}")

In [ ]:
# Cell 6: Transformer -- known-good control, via the existing harness (unchanged)
task_args_str = ",".join(f"{k}={v}" for k, v in TASK_ARGS.items())
!python src/train_task.py --config configs/recall/transformer_small.yaml \
    --task selective_copy --task_args {task_args_str} --steps {STEPS} \
    --batch_size {BATCH} --seed {SEED} --device cuda \
    --out {OUT}/transformer.json 2>&1 | tail -6

In [ ]:
# Cell 7: Our custom Mamba, post-fix, ssm_lr_scale=1 (default -- no boost)
!python src/train_task.py --config configs/recall/mamba_small.yaml \
    --task selective_copy --task_args {task_args_str} --steps {STEPS} \
    --batch_size {BATCH} --lr {LR} --seed {SEED} --device cuda \
    --out {OUT}/custom_mamba.json 2>&1 | tail -6

In [ ]:
# Cell 8: Official mamba-ssm oracle -- same task, same protocol, in-notebook
# training loop mirroring src/train_task.py exactly (masked loss, same LR
# schedule, same eval convention) since OracleMamba can't go through build_model().
import sys, math, time, json
sys.path.insert(0, "/kaggle/working/transformer-vs-ssm")
import torch
import torch.nn as nn
from src.tasks import IGNORE_INDEX, make_batcher

torch.manual_seed(SEED)
device = torch.device("cuda")
oracle = OracleMamba(vocab_size=TASK_ARGS["vocab_size"], n_layers=N_LAYERS,
                     d_model=D_MODEL, d_state=D_STATE, d_conv=D_CONV,
                     expand=EXPAND).to(device)
n_params = count_params(oracle)
print(f"OracleMamba: {n_params:,} params  "
      f"(reference: custom Mamba ~678k, transformer ~667k -- dimension-matched, "
      f"not param-matched; see decision table in the header)")

# Optimizer: respect the official module's own _no_weight_decay tags (A_log, D)
# rather than reimplementing src/optim.py's name-matching for a different module.
decay, no_decay = [], []
for n, p in oracle.named_parameters():
    (no_decay if getattr(p, "_no_weight_decay", False) else decay).append(p)
opt = torch.optim.AdamW(
    [{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],
    lr=LR, betas=(0.9, 0.95))
print(f"param groups: decay {sum(p.numel() for p in decay):,} | "
      f"no-decay {sum(p.numel() for p in no_decay):,}")

def get_lr(step, warmup=WARMUP, max_steps=STEPS, lr=LR):
    if step < warmup:
        return lr * (step + 1) / warmup
    prog = (step - warmup) / max(1, max_steps - warmup)
    return lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))

batcher = make_batcher("selective_copy", **TASK_ARGS)
val_tok, val_tgt, val_mask, _ = batcher(64, seed=10_000_000)  # same val-seed convention as train_task.py
val_tok = torch.from_numpy(val_tok).to(device)
val_tgt = torch.from_numpy(val_tgt).to(device)
val_mask = torch.from_numpy(val_mask).to(device)

t0 = time.time()
oracle.train()
for step in range(STEPS):
    lr = get_lr(step)
    for g in opt.param_groups:
        g["lr"] = lr
    tok, tgt, _, _ = batcher(BATCH, seed=step + SEED * 10_000_000)  # same train-seed convention
    tok = torch.from_numpy(tok).to(device)
    tgt = torch.from_numpy(tgt).to(device)
    logits = oracle(tok)
    loss = nn.functional.cross_entropy(logits.reshape(-1, TASK_ARGS["vocab_size"]),
                                       tgt.reshape(-1), ignore_index=IGNORE_INDEX)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(oracle.parameters(), 1.0)
    opt.step()

    if (step + 1) % 250 == 0 or step == STEPS - 1:
        oracle.eval()
        with torch.no_grad():
            vlogits = oracle(val_tok)
            vloss = nn.functional.cross_entropy(
                vlogits.reshape(-1, TASK_ARGS["vocab_size"]), val_tgt.reshape(-1),
                ignore_index=IGNORE_INDEX)
            vacc = (vlogits.argmax(-1)[val_mask] == val_tgt[val_mask]).float().mean().item()
        oracle.train()
        print(f"  step {step+1:>5d}  train_loss {loss.item():.4f}  "
              f"val_loss {vloss.item():.4f}  val_acc {vacc:.4f}  "
              f"({time.time()-t0:.0f}s)", flush=True)

record = {"arch": "mamba_ssm_oracle", "n_params": n_params, "final_val_acc": vacc,
         "final_val_loss": vloss.item(), "steps_run": STEPS, "seed": SEED,
         "wall_clock_s": round(time.time() - t0, 1),
         "chance_acc": 1.0 / ((TASK_ARGS["vocab_size"] - 1) // 2)}
json.dump(record, open(f"{OUT}/oracle_mamba_ssm.json", "w"), indent=2)
print(f"\nfinal val_acc {vacc:.4f}  (chance ~{record['chance_acc']:.4f})")

In [ ]:
# Cell 9: Verdict -- apply the decision table
import glob, json

results = {}
for f in glob.glob(f"{OUT}/*.json"):
    r = json.load(open(f))
    results[r.get("arch", r.get("task", "unknown"))] = r

chance = 1.0 / ((TASK_ARGS["vocab_size"] - 1) // 2)
tra = results.get("transformer", {}).get("final_val_acc", 0.0)
cus = results.get("mamba", {}).get("final_val_acc", 0.0)
ora = results.get("mamba_ssm_oracle", {}).get("final_val_acc", 0.0)

print(f"{'run':22s} {'val_acc':>9s}   (chance {chance:.3f})")
print("-" * 42)
print(f"{'transformer (control)':22s} {tra:>9.4f}")
print(f"{'custom mamba (post-fix)':22s} {cus:>9.4f}")
print(f"{'mamba-ssm oracle':22s} {ora:>9.4f}")

LEARN = 0.5  # generous threshold -- either passes cleanly or it doesn't at this task
print()
if tra < LEARN:
    print("INCONCLUSIVE -- even the transformer control failed. Task/recipe is at "
          "fault here, not either Mamba implementation. Fix the harness before "
          "drawing any conclusion about Mamba.")
elif cus >= LEARN and ora >= LEARN:
    print("Custom Mamba LEARNS, oracle LEARNS -- implementation validated. "
          "Proceed to MQAR.")
elif cus < LEARN and ora >= LEARN:
    print("Custom Mamba FAILS, oracle LEARNS -- a model-level discrepancy remains "
          "in the custom implementation (wrapper, precision, or training path). "
          "Diff against OracleBlock/OfficialMamba next, not another init/LR pass.")
elif cus < LEARN and ora < LEARN:
    print("Custom Mamba FAILS, oracle FAILS -- stop debugging the architecture. "
          "Investigate the task/training recipe itself (steps, batch, LR schedule, "
          "task difficulty at this scale) before touching either model again.")
else:
    print("Custom Mamba LEARNS, oracle FAILS -- harness/config mismatch between "
          "the two paths. Don't interpret either result until that's resolved.")